# IEEE-CIS Fraud Detection – Data Cleaning & Feature Engineering

In [54]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
print("Libraries loaded.")

Libraries loaded.


## 1. Load raw data and merge

In [55]:
import pandas as pd
import numpy as np
from pathlib import Path

TRAIN_TRANS_PATH = r"D:\project\data\train_transaction.csv"
TRAIN_ID_PATH    = r"D:\project\data\train_identity.csv"
TEST_TRANS_PATH  = r"D:\project\data\test_transaction.csv"
TEST_ID_PATH     = r"D:\project\data\test_identity.csv"

# thư mục lưu file processed
OUTPUT_DIR = Path(r"D:\project\data\merge")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading data...")

train_trans = pd.read_csv(TRAIN_TRANS_PATH)
train_id    = pd.read_csv(TRAIN_ID_PATH)
test_trans  = pd.read_csv(TEST_TRANS_PATH)
test_id     = pd.read_csv(TEST_ID_PATH)

train_df = train_trans.merge(train_id, on="TransactionID", how="left")
test_df  = test_trans.merge(test_id, on="TransactionID", how="left")

print("Merged train shape:", train_df.shape)
print("Merged test shape :", test_df.shape)
print("Output dir:", OUTPUT_DIR)

Loading data...
Merged train shape: (590540, 434)
Merged test shape : (506691, 433)
Output dir: D:\project\data\merge


## 2. Check missing values values

In [56]:
missing_train = train_df.isna().mean().sort_values(ascending=False)
missing_test = test_df.isna().mean().sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "train_missing_ratio": missing_train,
    "test_missing_ratio": missing_test
}).fillna(0)

print(missing_summary.head(20))
print("\nColumns with > 90% missing in train:", int((missing_summary["train_missing_ratio"] > 0.9).sum()))
print("Columns with > 90% missing in test :", int((missing_summary["test_missing_ratio"] > 0.9).sum()))

     train_missing_ratio  test_missing_ratio
C1              0.000000            0.000006
C10             0.000000            0.000006
C11             0.000000            0.000006
C12             0.000000            0.000006
C13             0.000000            0.009371
C14             0.000000            0.000006
C2              0.000000            0.000006
C3              0.000000            0.000006
C4              0.000000            0.000006
C5              0.000000            0.000006
C6              0.000000            0.000006
C7              0.000000            0.000006
C8              0.000000            0.000006
C9              0.000000            0.000006
D1              0.002149            0.011903
D10             0.128733            0.024759
D11             0.472935            0.348374
D12             0.890410            0.863321
D13             0.895093            0.756491
D14             0.894695            0.772654

Columns with > 90% missing in train: 12
Columns with >

## 3. remove missing 

In [57]:
high_missing_cols = missing_summary.index[missing_summary["train_missing_ratio"] > 0.90].tolist()
high_missing_cols = [c for c in high_missing_cols if c not in ["isFraud", "TransactionID"]]

train_df = train_df.drop(columns=high_missing_cols, errors="ignore")
test_df  = test_df.drop(columns=high_missing_cols, errors="ignore")

print("Dropped high-missing columns:", len(high_missing_cols))
print("Sample dropped columns:", high_missing_cols[:20])
print("New train shape:", train_df.shape)
print("New test shape :", test_df.shape)

Dropped high-missing columns: 12
Sample dropped columns: ['D7', 'dist2', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']
New train shape: (590540, 422)
New test shape : (506691, 431)


## 4. split train and test


In [58]:
TARGET_COL = "isFraud"
ID_COL = "TransactionID"

y = train_df[TARGET_COL].copy()
X = train_df.drop(columns=[TARGET_COL]).copy()
X_test = test_df.copy()

print("X shape     :", X.shape)
print("y shape     :", y.shape)
print("X_test shape:", X_test.shape)
print("Fraud ratio :", float(y.mean()))

X shape     : (590540, 421)
y shape     : (590540,)
X_test shape: (506691, 431)
Fraud ratio : 0.03499000914417313


## 5. Feature engineering 


In [59]:
def add_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Amount features
    if "TransactionAmt" in df.columns:
        amt = pd.to_numeric(df["TransactionAmt"], errors="coerce")
        df["TransactionAmt_log1p"] = np.log1p(amt.clip(lower=0))
        df["TransactionAmt_is_zero"] = (amt.fillna(0) == 0).astype(int)
        df["TransactionAmt_round"] = amt.fillna(-1).round(0)

    # Time features
    if "TransactionDT" in df.columns:
        dt = pd.to_numeric(df["TransactionDT"], errors="coerce").fillna(0)
        df["TransactionHour"] = ((dt // 3600) % 24).astype(int)
        df["TransactionDay"] = ((dt // (3600 * 24)) % 7).astype(int)
        df["TransactionWeek"] = ((dt // (3600 * 24 * 7))).astype(int)

    # Email match
    if "P_emaildomain" in df.columns and "R_emaildomain" in df.columns:
        df["email_match"] = (
            df["P_emaildomain"].fillna("missing").astype(str)
            == df["R_emaildomain"].fillna("missing").astype(str)
        ).astype(int)

    # Product + device interaction
    if "ProductCD" in df.columns and "DeviceType" in df.columns:
        df["product_device_combo"] = (
            df["ProductCD"].fillna("missing").astype(str) + "_" +
            df["DeviceType"].fillna("missing").astype(str)
        )

    # UID features
    if all(c in df.columns for c in ["card1", "card2"]):
        df["uid_card12"] = (
            df["card1"].fillna(-1).astype(str) + "_" +
            df["card2"].fillna(-1).astype(str)
        )

    if all(c in df.columns for c in ["card1", "card2", "addr1"]):
        df["uid_card_addr"] = (
            df["card1"].fillna(-1).astype(str) + "_" +
            df["card2"].fillna(-1).astype(str) + "_" +
            df["addr1"].fillna(-1).astype(str)
        )

    if "card1" in df.columns and "P_emaildomain" in df.columns:
        df["uid_card_email"] = (
            df["card1"].fillna(-1).astype(str) + "_" +
            df["P_emaildomain"].fillna("missing").astype(str)
        )

    if all(c in df.columns for c in ["card1", "addr1", "DeviceInfo"]):
        df["uid_card_addr_device"] = (
            df["card1"].fillna(-1).astype(str) + "_" +
            df["addr1"].fillna(-1).astype(str) + "_" +
            df["DeviceInfo"].fillna("missing").astype(str)
        )

    return df

X = add_basic_features(X)
X_test = add_basic_features(X_test)

print("Feature engineering done.")
print("X shape     :", X.shape)
print("X_test shape:", X_test.shape)

Feature engineering done.
X shape     : (590540, 433)
X_test shape: (506691, 443)


## 6. Split train / validation
Split có stratify theo `isFraud` để giữ tỷ lệ fraud ổn định.

In [60]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("Fraud ratio train:", float(y_train.mean()))
print("Fraud ratio val  :", float(y_val.mean()))

X_train: (472432, 433)
X_val  : (118108, 433)
Fraud ratio train: 0.03498916246147594
Fraud ratio val  : 0.0349933958749619


## 7. Frequency encoding


In [61]:
freq_cols = [c for c in [
    "card1", "card2", "card3", "card5",
    "addr1", "addr2",
    "P_emaildomain", "R_emaildomain",
    "ProductCD", "DeviceType", "DeviceInfo",
    "uid_card12", "uid_card_addr", "uid_card_email", "uid_card_addr_device",
    "product_device_combo"
] if c in X_train.columns]

freq_maps = {}

for col in freq_cols:
    freq_map = X_train[col].fillna("missing").astype(str).value_counts(dropna=False).to_dict()
    freq_maps[col] = freq_map

    X_train[f"{col}_freq"] = X_train[col].fillna("missing").astype(str).map(freq_map).fillna(0)
    X_val[f"{col}_freq"]   = X_val[col].fillna("missing").astype(str).map(freq_map).fillna(0)
    X_test[f"{col}_freq"]  = X_test[col].fillna("missing").astype(str).map(freq_map).fillna(0)

print("Frequency encoding added for", len(freq_cols), "columns")

Frequency encoding added for 16 columns


## 8. Amount group features


In [62]:
group_cols = [c for c in ["card1", "uid_card12", "uid_card_addr", "uid_card_email"] if c in X_train.columns]
group_amt_maps = {}

if "TransactionAmt" in X_train.columns:
    for col in group_cols:
        grp_mean = X_train.groupby(col)["TransactionAmt"].mean().to_dict()
        group_amt_maps[col] = grp_mean

        X_train[f"{col}_amt_mean"] = X_train[col].map(grp_mean).fillna(0)
        X_val[f"{col}_amt_mean"]   = X_val[col].map(grp_mean).fillna(0)
        X_test[f"{col}_amt_mean"]  = X_test[col].map(grp_mean).fillna(0)

        X_train[f"{col}_amt_ratio"] = X_train["TransactionAmt"] / (X_train[f"{col}_amt_mean"] + 1e-6)
        X_val[f"{col}_amt_ratio"]   = X_val["TransactionAmt"] / (X_val[f"{col}_amt_mean"] + 1e-6)
        X_test[f"{col}_amt_ratio"]  = X_test["TransactionAmt"] / (X_test[f"{col}_amt_mean"] + 1e-6)

    print("Transaction amount group features added.")
else:
    print("TransactionAmt not found, skipped.")

Transaction amount group features added.


## 9. Xác định categorical columns

In [63]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

print("Categorical columns:", len(cat_cols))
print("Numerical columns  :", len(num_cols))
print("Sample categorical:", cat_cols[:20])

Categorical columns: 34
Numerical columns  : 423
Sample categorical: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30']


## 10. Encode categorical features using OrdinalEncoder


In [64]:
from sklearn.preprocessing import OrdinalEncoder

# Chỉ giữ các cột categorical cùng tồn tại ở cả train/val/test
cat_cols = [
    c for c in cat_cols
    if c in X_train.columns and c in X_val.columns and c in X_test.columns
]

print("Number of categorical columns used:", len(cat_cols))

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

if len(cat_cols) > 0:
    X_train[cat_cols] = encoder.fit_transform(
        X_train[cat_cols].fillna("missing").astype(str)
    )
    X_val[cat_cols] = encoder.transform(
        X_val[cat_cols].fillna("missing").astype(str)
    )
    X_test[cat_cols] = encoder.transform(
        X_test[cat_cols].fillna("missing").astype(str)
    )
    print("Ordinal encoding completed.")
else:
    print("No categorical columns found.")

Number of categorical columns used: 21
Ordinal encoding completed.


## 11. split into train/test/valdation

In [65]:
X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=np.nan)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=np.nan)
X_val = X_val.reindex(columns=X_train.columns, fill_value=np.nan)
X_test = X_test.reindex(columns=X_train.columns, fill_value=np.nan)

print("Aligned shapes:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

Aligned shapes:
X_train: (472432, 457)
X_val  : (118108, 457)
X_test : (506691, 457)


## 12. fill numeric and missing by median  train


In [66]:
X_train = X_train.apply(pd.to_numeric, errors="coerce")
X_val   = X_val.apply(pd.to_numeric, errors="coerce")
X_test  = X_test.apply(pd.to_numeric, errors="coerce")

train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians)
X_val   = X_val.fillna(train_medians)
X_test  = X_test.fillna(train_medians)

X_train = X_train.fillna(0)
X_val   = X_val.fillna(0)
X_test  = X_test.fillna(0)

print("Any NaN in X_train:", int(X_train.isna().sum().sum()))
print("Any NaN in X_val  :", int(X_val.isna().sum().sum()))
print("Any NaN in X_test :", int(X_test.isna().sum().sum()))

Any NaN in X_train: 0
Any NaN in X_val  : 0
Any NaN in X_test : 0


## 13. removet TransactionID  out of feature


In [67]:
drop_feature_cols = [c for c in ["TransactionID"] if c in X_train.columns]

X_train = X_train.drop(columns=drop_feature_cols, errors="ignore")
X_val   = X_val.drop(columns=drop_feature_cols, errors="ignore")
X_test  = X_test.drop(columns=drop_feature_cols, errors="ignore")

print("Dropped from features:", drop_feature_cols)
print("Final feature count:", X_train.shape[1])

Dropped from features: ['TransactionID']
Final feature count: 456


## 14. create processed datasets

In [68]:
train_processed = X_train.copy()
train_processed[TARGET_COL] = y_train.values

val_processed = X_val.copy()
val_processed[TARGET_COL] = y_val.values

test_processed = X_test.copy()

print("train_processed:", train_processed.shape)
print("val_processed  :", val_processed.shape)
print("test_processed :", test_processed.shape)

train_processed: (472432, 457)
val_processed  : (118108, 457)
test_processed : (506691, 456)


## 15. Save CSV files

In [69]:
train_out = OUTPUT_DIR / "train_processed.csv"
val_out   = OUTPUT_DIR / "val_processed.csv"
test_out  = OUTPUT_DIR / "test_processed.csv"

train_processed.to_csv(train_out, index=False)
val_processed.to_csv(val_out, index=False)
test_processed.to_csv(test_out, index=False)

print("Saved:")
print(train_out)
print(val_out)
print(test_out)

Saved:
D:\project\data\merge\train_processed.csv
D:\project\data\merge\val_processed.csv
D:\project\data\merge\test_processed.csv


## 16. Save metadata and training artifacts

In [70]:
metadata = {
    "target_col": TARGET_COL,
    "id_col": ID_COL,
    "dropped_high_missing_cols": high_missing_cols,
    "categorical_columns_before_encoding": cat_cols,
    "final_feature_columns": X_train.columns.tolist(),
    "n_features": int(X_train.shape[1]),
    "train_shape": list(train_processed.shape),
    "val_shape": list(val_processed.shape),
    "test_shape": list(test_processed.shape),
    "fraud_ratio_train": float(y_train.mean()),
    "fraud_ratio_val": float(y_val.mean()),
    "freq_encoded_columns": freq_cols,
    "group_amount_columns": group_cols,
}

metadata_path = OUTPUT_DIR / "preprocessing_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

encoder_path = OUTPUT_DIR / "ordinal_encoder.pkl"
with open(encoder_path, "wb") as f:
    pickle.dump(encoder, f)

medians_path = OUTPUT_DIR / "train_medians.pkl"
with open(medians_path, "wb") as f:
    pickle.dump(train_medians.to_dict(), f)

freq_maps_path = OUTPUT_DIR / "frequency_maps.pkl"
with open(freq_maps_path, "wb") as f:
    pickle.dump(freq_maps, f)

group_amt_path = OUTPUT_DIR / "group_amount_maps.pkl"
with open(group_amt_path, "wb") as f:
    pickle.dump(group_amt_maps, f)

print("Saved metadata and preprocessing objects.")
print(metadata_path)
print(encoder_path)
print(medians_path)
print(freq_maps_path)
print(group_amt_path)

Saved metadata and preprocessing objects.
D:\project\data\merge\preprocessing_metadata.json
D:\project\data\merge\ordinal_encoder.pkl
D:\project\data\merge\train_medians.pkl
D:\project\data\merge\frequency_maps.pkl
D:\project\data\merge\group_amount_maps.pkl
